# 08 - Two-phase tuning flow end to end

This notebook confirms the two-phase contract the orchestrator implements
(`_tune_model` in `pipelines.tuning_pipeline.pipeline`): Phase 1 optimises
optimisation parameters, its decoded best is fed into Phase 2 via
`Phase2Tuner.best_phase1_params`, and Phase 2 then searches architecture
parameters on top of that. It exercises the real `Phase1Tuner` and
`Phase2Tuner` with synthetic objectives and known optima.

We confirm Phase 1 recovers its optimal lr, that this value is carried into the
Phase-2 config, and that Phase 2 recovers its optimal architecture choice.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt
import optuna

import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.figsize" : (7.0, 4.2),
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
})


## Mock tuners for both phases

Both subclasses reuse the genuine `_apply_params` so the override paths are real;
only the loss is synthetic. Phase 1 targets `encoder_lr = 1e-3`; Phase 2 targets
the `gelu` activation while reading back the Phase-1 lr.

In [ ]:
from pipelines.tuning_pipeline.tuners import Phase1Tuner, Phase2Tuner
from configuration.models_config    import UNetConfig
from configuration.tuning_config     import Phase1TuneConfig, Phase2TuneConfig

LR_STAR     = 1e-3
ACT_STAR    = 'gelu'
act_choices = UNetConfig.tunable_arch_params()['activation']['choices']

class NullLogger:
    def __getattr__(self, _):
        return lambda *a, **k: None

class MockPhase1(Phase1Tuner):
    def _objective(self, trial):
        cfg = self.model_config_cls()
        self._apply_params(trial, cfg)
        trial.set_user_attr('encoder_lr', cfg.encoder_lr)
        return float((np.log10(cfg.encoder_lr) - np.log10(LR_STAR)) ** 2)

class MockPhase2(Phase2Tuner):
    def _objective(self, trial):
        cfg = self.model_config_cls()
        self._apply_params(trial, cfg)
        trial.set_user_attr('activation', cfg.activation)
        trial.set_user_attr('encoder_lr', cfg.encoder_lr)
        act_pen = 0.0 if cfg.activation == ACT_STAR else 1.0
        return float(act_pen + 0.01 * np.random.default_rng(trial.number).random())

common = dict(
    model_name          = 'unet',
    model_config_cls    = UNetConfig,
    base_trainer_config = None,
    base_dataset_config = None,
    log_dir             = '/tmp/nb_tuning_unused',
    logger              = NullLogger(),
)
print('mock tuners defined')

## Phase 1: recover the optimal learning rate

Run Phase 1 and decode its best trial the way the orchestrator does.

In [ ]:
p1 = MockPhase1(tune_cfg=Phase1TuneConfig(), **common)

study1 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10))
p1.run(study1, n_trials=100)

best_p1_lr   = study1.best_trial.user_attrs['encoder_lr']
decoded_p1   = {'encoder_lr': best_p1_lr}
print('Phase-1 best encoder_lr:', best_p1_lr, '(target %.0e)' % LR_STAR)

## Phase 2: search architecture on top of the Phase-1 best

The decoded Phase-1 dict is passed as `best_phase1_params`. We then confirm,
inside one Phase-2 trial, that the carried-over `encoder_lr` is present in the
config alongside the freshly sampled architecture fields.

In [ ]:
p2 = MockPhase2(tune_cfg=Phase2TuneConfig(), best_phase1_params=decoded_p1, **common)

study2 = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10))
p2.run(study2, n_trials=100)

best_act = study2.best_trial.user_attrs['activation']
carried  = study2.best_trial.user_attrs['encoder_lr']
print('Phase-2 best activation :', best_act, '(target %s)' % ACT_STAR)
print('carried encoder_lr      :', carried)
print('carried matches P1 best :', np.isclose(carried, best_p1_lr))

## Visual summary of the two phases

Left: Phase-1 best-so-far lr distance to target. Right: how often each
activation was the sampled choice in Phase 2, with the target highlighted.

In [ ]:
from collections import Counter

p1_best_curve = np.minimum.accumulate([t.value for t in study1.trials])
act_counts    = Counter(t.user_attrs['activation'] for t in study2.trials)

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4))

a0.plot(p1_best_curve, color='steelblue', lw=2)
a0.set_yscale('log')
a0.set_xlabel('trial')
a0.set_ylabel('best-so-far (log-lr distance)^2')
a0.set_title('Phase 1 convergence')

bars   = [act_counts.get(a, 0) for a in act_choices]
colors = ['darkorange' if a == ACT_STAR else 'steelblue' for a in act_choices]
a1.bar(act_choices, bars, color=colors)
a1.set_ylabel('times sampled')
a1.set_title('Phase 2 activation sampling (target in orange)')
a1.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()

## Expected visual outcome

Phase 1's best learning rate sits close to `1e-3` and its convergence curve
drops steadily. Phase 2 selects `gelu` as the best activation and the carried
`encoder_lr` matches the Phase-1 best (printed `True`). The activation bar chart
shows the target activation sampled most often once TPE concentrates on it.